# Day 2 – Data Preparation (VOLTA)

**Goal:** turn the raw energy and weather files into one clean hourly table, ready for modeling.
**Output:** `data/processed/volta_clean_hourly.csv` (35,064 rows, one per hour, UTC).

**Rules we followed**
- Raw files in `data/raw/` are never modified. Every step works on a copy.
- We never delete rows: the hourly sequence must stay complete for the windows.
- Impossible values become NaN, then short gaps are filled by time interpolation (max 24 h), per city / per column.
- The fixed limits come from physics, not from the data, so there is no leakage from validation or test.
- The four targets: `total load actual`, `price actual`, `renewable_total`, `fossil_total`. The price is never changed.

**Sections**
1. Load  2. Weather: duplicates and description columns  3. Weather: impossible readings
4. Energy: zeros in gas and nuclear  5. Energy: where are the gaps?  6. Energy: empty columns and gap filling
7. Energy: three hours where the whole generation record failed  8. Derived targets  9. Merge
10. Save and check  11. Known issue: pressure in Jan–Jun 2015  12. Columns we will not use in models

## 1. Load

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
OUT = Path("../data/processed")

energy_raw  = pd.read_csv(RAW / "energy_dataset.csv")
weather_raw = pd.read_csv(RAW / "weather_features.csv")

# work on copies: the raw tables stay untouched
energy  = energy_raw.copy()
weather = weather_raw.copy()

energy["time"]    = pd.to_datetime(energy["time"], utc=True)
weather["dt_iso"] = pd.to_datetime(weather["dt_iso"], utc=True)
energy = energy.sort_values("time").set_index("time")

print("Energy:", energy.shape, "| Weather:", weather.shape)   # (35064, 28) | (178396, 17)

## 2. Weather: description columns and duplicates

The weather file has more than one row for some (city, hour). The duplicated rows have identical measurements and differ only in the text description of the weather (for example "light rain" and "drizzle" in the same hour).
We do not use the description columns, so we drop them. After that the duplicates are identical rows and can be removed safely.

In [ ]:
desc_cols = ["weather_id", "weather_main", "weather_description", "weather_icon"]
weather = weather.drop(columns=desc_cols)

before = len(weather)
weather = weather.drop_duplicates()          # identical rows only
print("removed rows:", before - len(weather))   # 3076

print(weather.groupby("city_name").size())      # 35064 per city
print("still duplicated (city, time):", weather.duplicated(["city_name", "dt_iso"]).sum())   # 0

## 3. Weather: impossible readings

Three kinds of impossible readings were found in the Day 1 quality check:
- pressure outside 900–1100 hPa (Barcelona only),
- humidity = 0 (Barcelona and Madrid),
- wind speed >= 28.5 m/s, a violent storm (Valencia only).

They are sensor faults: they appear in one city while the other cities are normal, they cluster in early 2015, and one isolated spike reaches 133 m/s.
**Decision (agreed with Marta):** the value becomes NaN, the row stays, and the gap is filled by time interpolation per city, with a maximum of 24 hours.

In [ ]:
weather = weather.sort_values(["city_name", "dt_iso"]).reset_index(drop=True)
weather_clean = weather.copy()

rules = {
    "pressure":   lambda s: ~s.between(900, 1100) & s.notna(),
    "humidity":   lambda s: s == 0,
    "wind_speed": lambda s: s >= 28.5,
}

# 1) impossible -> NaN
flagged = {}
for col, rule in rules.items():
    bad = rule(weather_clean[col])
    flagged[col] = int(bad.sum())
    weather_clean.loc[bad, col] = np.nan
print("set to NaN:", flagged)          # pressure 48, humidity 63, wind_speed 12

# 2) time interpolation, each city on its own (max gap 24 h)
for col in rules:
    weather_clean[col] = (
        weather_clean.set_index("dt_iso")
        .groupby("city_name")[col]
        .transform(lambda s: s.interpolate(method="time", limit=24, limit_area="inside"))
        .values
    )

# 3) check
print("NaN left:", weather_clean[list(rules)].isna().sum().to_dict())     # all 0
changed = {c: int((weather[c] != weather_clean[c]).sum()) for c in rules}
print("values changed:", changed)                                         # same counts as above

## 4. Energy: zeros in gas and nuclear

Gas and nuclear plants run continuously, so a value of exactly 0 is a data error.
Found: gas = 0 in one hour (2018-06-02 09:00), nuclear = 0 in three hours (2017-11-12 20:00, 2017-11-14 11:00 and 18:00). Each zero is isolated, so the average of the hour before and the hour after is a good estimate (Marta's recommendation).

In [ ]:
energy_clean = energy.copy()

for c in ["generation fossil gas", "generation nuclear"]:
    n0 = int((energy_clean[c] == 0).sum())
    energy_clean.loc[energy_clean[c] == 0, c] = np.nan
    energy_clean[c] = energy_clean[c].interpolate(method="time", limit=24, limit_area="inside")
    print(c, "| zeros fixed:", n0, "| zeros left:", int((energy_clean[c] == 0).sum()))   # 1 and 3

print(energy_clean.loc["2018-06-02 08:00":"2018-06-02 10:00", "generation fossil gas"])   # 09:00 -> 6905.5

## 5. Energy: where are the gaps?

Before filling anything we check how long the gaps are and where they are relative to the split boundaries (train 2015–2016, validation 2017, test 2018).
Interpolation uses the value before and after a gap, so a gap on a boundary could pull information across splits.

In [ ]:
def longest_run(mask):
    runs = mask.ne(mask.shift()).cumsum()[mask].value_counts()
    return (int(runs.max()) if len(runs) else 0, len(runs))

rows = []
for c in energy_clean.columns:
    m = energy_clean[c].isna()
    if m.any() and m.sum() < len(m):
        longest, n = longest_run(m)
        rows.append((c, int(m.sum()), longest, n))
pd.DataFrame(rows, columns=["column", "NaN", "longest_gap_h", "gaps"]).sort_values("longest_gap_h", ascending=False)

In [ ]:
b1 = pd.Timestamp("2017-01-01", tz="UTC")     # train | validation
b2 = pd.Timestamp("2018-01-01", tz="UTC")     # validation | test

gap_cols = [c for c in energy_clean.columns
            if energy_clean[c].isna().any() and not energy_clean[c].isna().all()]
na_rows = energy_clean.index[energy_clean[gap_cols].isna().any(axis=1)]

print("NaN hours total:", len(na_rows))                                   # 47
print("per split:", pd.Series(np.where(na_rows < b1, "train",
                              np.where(na_rows < b2, "val", "test"))).value_counts().to_dict())   # train 43, val 2, test 2
near = [t for t in na_rows if abs(t - b1) <= pd.Timedelta("12h") or abs(t - b2) <= pd.Timedelta("12h")]
print("NaN hours within 12h of a split boundary:", near)                  # []

## 6. Energy: empty columns and gap filling

- Two columns are 100% empty (`generation hydro pumped storage aggregated`, `forecast wind offshore eday ahead`): they carry no information, so they are removed.
- All other gaps are short (longest 8 hours, limit 24) and none is near a split boundary, so they are filled by time interpolation.

In [ ]:
empty_cols = [c for c in energy_clean.columns if energy_clean[c].isna().all()]
print("empty columns dropped:", empty_cols)
energy_clean = energy_clean.drop(columns=empty_cols)

fill_cols = [c for c in energy_clean.columns if energy_clean[c].isna().any()]
before = energy_clean.isna().sum()
energy_clean[fill_cols] = energy_clean[fill_cols].interpolate(method="time", limit=24, limit_area="inside")

print("NaN left:", int(energy_clean.isna().sum().sum()))                  # 0
print("filled per column:\n", (before - energy_clean.isna().sum())[lambda s: s > 0])
print("shape:", energy_clean.shape)                                        # (35064, 26)

## 7. Energy: three hours where the whole generation record failed

While checking the renewable total we found 3 hours with a total of exactly 0, which is impossible.
These are the same hours as the nuclear zeros. In those hours **all generation columns are 0** (the total load is also missing in two of them), so the record for the whole row failed. Fixing nuclear alone left the other columns at 0.
**Decision:** all generation columns in those 3 hours become NaN and are filled by interpolation. All three hours are in the validation set (Nov 2017) and their neighbours are in the same set.

In [ ]:
bad_hours = pd.to_datetime(["2017-11-12 20:00", "2017-11-14 11:00", "2017-11-14 18:00"], utc=True)
gen_cols = [c for c in energy.columns if c.startswith("generation")]

# evidence (original data): which generation columns are 0 in all three hours
all_zero = (energy.loc[bad_hours, gen_cols] == 0).all()
print("generation columns that are 0 in all 3 hours:", int(all_zero.sum()), "of", len(gen_cols))
print(energy.loc[bad_hours, ["total load actual", "price actual"]])

In [ ]:
gen_cols_clean = [c for c in energy_clean.columns if c.startswith("generation")]

energy_clean.loc[bad_hours, gen_cols_clean] = np.nan
energy_clean[gen_cols_clean] = energy_clean[gen_cols_clean].interpolate(
    method="time", limit=24, limit_area="inside")

print("NaN left:", int(energy_clean.isna().sum().sum()))    # 0
print(energy_clean.loc[bad_hours, ["generation nuclear", "generation wind onshore", "generation fossil gas"]])

## 8. Derived targets: renewable and fossil totals

- **Renewable** = 6 columns: biomass, hydro run-of-river and poundage, hydro water reservoir, other renewable, solar, wind onshore.
- **Fossil** = 4 columns: brown coal/lignite, gas, hard coal, oil.
- `min_count` makes the total NaN if any component is missing (a plain sum would count a missing value as 0).
They are computed after all the fixes above.

In [ ]:
renew_cols  = ["generation biomass",
               "generation hydro run-of-river and poundage",
               "generation hydro water reservoir",
               "generation other renewable",
               "generation solar",
               "generation wind onshore"]
fossil_cols = ["generation fossil brown coal/lignite",
               "generation fossil gas",
               "generation fossil hard coal",
               "generation fossil oil"]

energy_clean["renewable_total"] = energy_clean[renew_cols].sum(axis=1, min_count=len(renew_cols))
energy_clean["fossil_total"]    = energy_clean[fossil_cols].sum(axis=1, min_count=len(fossil_cols))

targets = ["total load actual", "price actual", "renewable_total", "fossil_total"]
print("NaN in targets:\n", energy_clean[targets].isna().sum())                    # all 0
print("price unchanged:", energy_clean["price actual"].equals(energy["price actual"]))   # True
print(energy_clean[targets].describe().round(1).T[["min", "mean", "max"]])        # renewable min 2128, fossil min 2673

## 9. Merge weather and energy

The weather table is long (one row per city and hour). We turn it into one column per city and measurement (for example `madrid_temp`) and join it to the energy table on the hour.

In [ ]:
w = weather_clean.copy()
w["city_name"] = w["city_name"].str.strip().str.lower()

wide = w.pivot(index="dt_iso", columns="city_name")
wide.columns = [f"{city}_{col}" for col, city in wide.columns]
wide.index.name = "time"

merged = energy_clean.join(wide, how="left")

print("shape:", merged.shape)                                          # (35064, 83)
print("NaN:", int(merged.isna().sum().sum()))                          # 0
print("unique + sorted:", merged.index.is_unique, merged.index.is_monotonic_increasing)   # True True
print("price unchanged:", merged["price actual"].equals(energy["price actual"]))          # True

## 10. Save and check

If a previous version of the file exists, the new table is compared with it first. If they differ, the new table is written to a separate `_NEW` file so nothing is overwritten by mistake.

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
path = OUT / "volta_clean_hourly.csv"

target_path = path
if path.exists():
    old = pd.read_csv(path, index_col="time", parse_dates=["time"])
    same = (old.shape == merged.shape and list(old.columns) == list(merged.columns)
            and np.allclose(old.values, merged.values))
    print("identical to the previous saved version:", same)
    if not same:
        target_path = OUT / "volta_clean_hourly_NEW.csv"
        print("DIFFERENT -> writing to", target_path.name, "instead of overwriting")

merged.to_csv(target_path)

chk = pd.read_csv(target_path, index_col="time", parse_dates=["time"])
print("saved:", target_path, "|", round(target_path.stat().st_size / 1e6, 1), "MB")
print("shape:", chk.shape, "| NaN:", int(chk.isna().sum().sum()))
print("index tz:", chk.index.tz, "| start:", chk.index.min(), "| end:", chk.index.max())
print("values equal to memory:", np.allclose(chk.values, merged.values))

## 11. Known issue: pressure in Jan–Jun 2015 (not changed in the file)

We compared Jan–Jun 2015 with Jan–Jun of 2016–2018 (same seasons), column by column. Only two columns are clearly off:

| column | std in 2015 vs usual | mean in 2015 vs usual |
|---|---|---|
| `madrid_pressure` | 3.1x | 981 vs 1015 hPa |
| `valencia_pressure` | 2.0x | 1009 vs 1015 hPa |

The values stay inside 900–1100 so the limits do not catch them, and six months is far too long to interpolate.
In the plot, Madrid and Valencia drop and jump in step-like patterns until the end of June 2015, while Seville and Bilbao stay stable.
**Decision:** the columns stay in the file but are excluded from the models (section 12).

In [ ]:
cities = ["barcelona", "bilbao", "madrid", "seville", "valencia"]
keys = ["temp", "pressure", "humidity", "wind_speed", "wind_deg", "clouds_all"]
wcols = [c for c in merged.columns if c.split("_", 1)[0] in cities and c.split("_", 1)[1] in keys]

h1 = merged.loc[merged.index.month <= 6, wcols]
g = h1.groupby(h1.index.year)
std, mean = g.std().T, g.mean().T
others = [2016, 2017, 2018]
cmp = pd.DataFrame({
    "std_2015":    std[2015].round(1),
    "std_others":  std[others].mean(axis=1).round(1),
    "std_ratio":   (std[2015] / std[others].mean(axis=1)).round(2),
    "mean_2015":   mean[2015].round(1),
    "mean_others": mean[others].mean(axis=1).round(1),
})
cmp.sort_values("std_ratio", ascending=False).head(8)

In [ ]:
import plotly.express as px
sub = merged.loc["2015-01-01":"2015-08-31", ["madrid_pressure", "valencia_pressure", "seville_pressure", "bilbao_pressure"]]
px.line(sub.resample("6h").mean(), title="Pressure Jan-Aug 2015 (6-hour mean)").show()

## 12. Columns we will not use in models

The saved file keeps all columns. This list is only for choosing model inputs, and every column has a reason:
- **Constant** (one value all the time, no information): 6 energy columns (`coal-derived gas`, `oil shale`, `peat`, `geothermal`, `marine`, `wind offshore`) and the snow columns.
- **Duplicate information:** `temp_min` and `temp_max` (correlation 0.97 with `temp`).
- **Almost always zero:** `rain_3h`, `snow_3h`.
- **Hard to use as is:** `wind_deg` (an angle, 0 and 360 are the same direction). We can come back to it later with sine and cosine.
- **Unreliable in Jan–Jun 2015:** `madrid_pressure`, `valencia_pressure`.

Still open (not excluded yet): `forecast solar day ahead`, `forecast wind onshore day ahead`, `total load forecast`, `price day ahead`. They are allowed as inputs only after we confirm they are the version published the day before.

In [ ]:
EXCLUDE_WEATHER_TYPES = ["temp_min", "temp_max", "rain_3h", "snow_3h", "wind_deg"]
exclude = [c for c in merged.columns
           if c.split("_", 1)[0] in cities and c.split("_", 1)[1] in EXCLUDE_WEATHER_TYPES]
exclude += ["madrid_pressure", "valencia_pressure"]
exclude += [c for c in merged.columns if merged[c].nunique() <= 1]          # constant columns
exclude = sorted(set(exclude))

model_cols = [c for c in merged.columns if c not in exclude]
print("columns in the file:", merged.shape[1], "| excluded:", len(exclude), "| usable:", len(model_cols))   # 83 | 33 | 50
print("excluded:", exclude)